In [9]:
import neuron as neuron

from dataProcessing import getData, getFilename, calculateLatency, calculateVelocity
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stimulationProtocols import getCOVIDFullTime
import json
from plot import plotLatency, plotRecoveryCycle
import os
from main_CMi import run


neuron.load_mechanisms('./MOD_Tigerholm')

Mechanisms already loaded from path: ./MOD_Tigerholm.  Aborting.


True

In [10]:
params_orig = {
    'gPump': -0.00485891709589456,
    'gNav17Parent': 0.22831806252579703,
    'gNav17': 0.33198932295899997,
    'gNav18Parent': 0.2411879110860178,
    'gNav18': 0.04520936848183172,
    'gNav19': 0.0032208034421239012,
    'gKs': 0.0030962055552702606,
    'gKf': 0.025994684685553986,
    'gH': 0.009167424690232623,
    'gKdr': 0.013875365047132446,
    'gKna': 0.0014444412024341238,
    'vRest': -55.0
}

# gNav17/gNav18 scaling is automatically applied to their Parent counterparts
# this runs custom parameter combinations instead of a grid search to save time. for full grid search, use analysis-COVID.ipynb.
experiments = [
    {}, # initial
    {"gNav17": 0.3, "gNav18": 0.30},
    {"gNav17": 0.2, "gNav18": 0.20, "gH": 0.20},
    {"gNav17": 0.2, "gNav18": 0.30, "gH": 0.30},
    {"gNav17": 0.3, "gNav18": 0.30, "gH": 0.40},
    {"gNav17": 0.2, "gNav18": 0.30, "gKdr": 0.20},
    {"gNav17": 0.3, "gNav18": 0.30, "gKs": -0.20},
    {"gNav17": 0.3, "gNav18": 0.30, "gKs": -0.20, "gKdr": 0.20},
    {"gNav17": 0.3, "gNav18": 0.30, "gKs": -0.20, "gKdr": 0.20, "gH": 0.40},
    {"gNav17": 0.3, "gNav18": 0.30, "gKs": -0.20, "gKdr": 0.20, "gH": 0.30},
    {"gNav17": 0.3, "gNav18": 0.30, "gKdr": 0.20, "gH": 0.30},
    {"gNav17": 0.3, "gNav18": 0.30, "gKs": -0.20, "gH": 0.30},
    #{"gNav17": 0.3, "gNav18": 0.35, "gKs": -0.20, "gKdr": 0.20, "gH": 0.40}, # gNav18 < +30%.
    {"gNav17": 0.3, "gNav18": 0.30, "gKs": -0.10, "gKdr": 0.10, "gH": 0.20}
]

PARENT_COUPLED = {"gNav17": "gNav17Parent", "gNav18": "gNav18Parent"}

protocol = 42

def apply_scales(base, scales):
    """Apply relative scaling factors to a copy of the baseline parameter dict."""
    p = base.copy()
    for name, dg in scales.items():
        p[name] = base[name] * (1 + dg)
        if name in PARENT_COUPLED:
            parent = PARENT_COUPLED[name]
            p[parent] = base[parent] * (1 + dg)
    return p



params = [apply_scales(params_orig, exp) for exp in experiments]
print(f"Loading {len(params)} simulations.")


Loading 13 simulations.


In [11]:
results = []
for param_dict in params:
    results.append(
        getData(
            prot=protocol,
            filetype="spikes",
            scalingFactor=0.1,
            gPump=param_dict['gPump'],
            gNav17=param_dict['gNav17'],
            gNav17Parent=param_dict['gNav17Parent'],
            gNav18=param_dict['gNav18'],
            gNav18Parent=param_dict['gNav18Parent'],
            gNav19=param_dict['gNav19'],
            gKs=param_dict['gKs'],
            gKf=param_dict['gKf'],
            gH=param_dict['gH'],
            gKdr=param_dict['gKdr'],
            gKna=param_dict['gKna'],
            vRest=param_dict['vRest']
        )
    )

In [12]:
getFilename(
            prot=protocol,
            filetype="spikes",
            scalingFactor=0.1,
            gPump=param_dict['gPump'],
            gNav17=param_dict['gNav17'],
            gNav17Parent=param_dict['gNav17Parent'],
            gNav18=param_dict['gNav18'],
            gNav18Parent=param_dict['gNav18Parent'],
            gNav19=param_dict['gNav19'],
            gKs=param_dict['gKs'],
            gKf=param_dict['gKf'],
            gH=param_dict['gH'],
            gKdr=param_dict['gKdr'],
            gKna=param_dict['gKna'],
            vRest=param_dict['vRest']
        )

'Results/spikes_Prot42_scale0.1_gPump-0.004859_gNav170.431586_gNav17P0.296813_gNav180.058772_gNav18P0.313544_gNav190.003221_gKs0.002787_gKf0.025995_gH0.011001_gKdr0.015263_gKna0.001444_vRest-55.0.csv'

In [13]:
data_stim = getData(prot=protocol, filetype="stim")

In [14]:
def get_metrics(data_aps, data_stim, Slow025HzStart=1, Slow025HzEnd=90, Fast2HzStart=90, Fast2HzEnd=450,
                Fast2HzPost30S=458):
    initial_velocity = calculateVelocity(data_aps, data_stim)[0]
    latency = calculateLatency(data_aps, data_stim, norm=False)[:, 1]
    latency_points = [latency[Slow025HzStart], latency[Fast2HzStart], latency[Fast2HzEnd], latency[Fast2HzPost30S]]
    Slow025StartToEnd = (latency[Slow025HzEnd] - latency[Slow025HzStart]) / latency[Slow025HzStart]
    Slow025EndToFast2HzEnd = (latency[Fast2HzEnd] - latency[Slow025HzEnd]) / latency[Slow025HzEnd]
    # recovery at 30 s (latency at 30 s after 2 Hz stimulation compared to latency before 0.25 Hz stimulation)
    # can be also negative (but unlikely)
    Fast2HzStartToPost30S = (latency[Fast2HzPost30S] - latency[Slow025HzStart]) / latency[Slow025HzStart]
    TimeTo50Percent = 0  # not implemented yet because it is barely changed by the conductancies

    recovery_50_percent_threshold = (latency[Fast2HzStart] + latency[Fast2HzEnd]) / 2

    for n in np.arange(Fast2HzEnd, SimulationEnd):
        # if the latency at number t-th spike is lower than the 50 % recovery threshold
        if latency[n] < recovery_50_percent_threshold:
            # use the time as the time until 50 % recovery
            TimeTo50Percent = getCOVIDFullTime(n) - getCOVIDFullTime(
                Fast2HzStart)  # can be made more precise with linear interpolation
            break

    return (initial_velocity, Slow025StartToEnd, Slow025EndToFast2HzEnd, Fast2HzStartToPost30S, TimeTo50Percent,
            latency_points)

In [15]:
Slow025HzStart = 1
Slow025HzEnd = 90
Fast2HzStart = 90  # 90 stimulations at 0.25 Hz initially
Fast2HzEnd = 450  # 360 stimulations at 2 Hz
Fast2HzPost30S = Fast2HzEnd + 8  # 32 s after reducing the stimulation frequency from 2 Hz to 0.25 Hz again

SimulationEnd = data_stim.shape[0] - 1 # ugly but works
points = [Slow025HzStart, Slow025HzEnd, Fast2HzStart, Fast2HzEnd, Fast2HzPost30S, SimulationEnd]
points_name = ["InitialVelocity", "Slow025HzStart", "Slow025HzEnd", "Fast2HzStart", "Fast2HzEnd", "Fast2HzPost30S",
               "SimulationEnd"]
slowing_name = [points_name[0], points_name[1] + "-" + points_name[2], points_name[3] + "-" + points_name[4],
                points_name[5], "TimeTo50Percent"]

In [16]:
metrics = []
metrics_normalized = []
for result in results:
    metric = get_metrics(result, data_stim)[:-1]
    metrics.append(np.array(metric))

    metric_normalized = []
    for i, m in enumerate(metric):
        metric_normalized.append(m / metrics[0][i])
    metrics_normalized.append(np.array(metric_normalized))

In [17]:
print("Normalized metrics:")
print(metrics_normalized)

Normalized metrics:
[array([1., 1., 1., 1., 1.]), array([1.09686132, 1.19357151, 0.91498661, 0.96286149, 0.9875    ]), array([1.06070773, 1.10434041, 0.94162342, 0.96431099, 0.9625    ]), array([1.06930193, 1.15180822, 0.934533  , 0.95927966, 0.95      ]), array([1.08291311, 1.13287136, 0.91471354, 0.94255413, 0.95      ]), array([1.06864957, 1.33117459, 0.96967215, 1.02102053, 0.95      ]), array([1.09820233, 1.36179555, 0.90467914, 0.99161375, 1.0875    ]), array([1.08695446, 1.52744013, 0.93649233, 1.03658174, 1.        ]), array([1.0732517 , 1.40977039, 0.96010101, 1.0266807 , 0.9375    ]), array([1.07651492, 1.44749016, 0.95387577, 1.02862783, 0.95      ]), array([1.07551061, 1.26137774, 0.96089163, 0.99646088, 0.925     ]), array([1.0874354 , 1.30143216, 0.91439936, 0.97954606, 1.        ]), array([1.08489174, 1.30565016, 0.93725539, 0.994252  , 0.9625    ])]


In [18]:
experiment_factors =  [1.20930233, 1.34963325, 0.88333333, 1.60406091, 0.64656965]

Number 6 and number 9 are the best fits, with the 6th one being the best.

In [20]:
print(metrics_normalized[5])

[1.06864957 1.33117459 0.96967215 1.02102053 0.95      ]


In [21]:
{"gNav17": 0.2, "gNav18": 0.30, "gKdr": 0.20}

{'gNav17': 0.2, 'gNav18': 0.3, 'gKdr': 0.2}